<a href="https://colab.research.google.com/github/BillJr99/Ursinus-CS357-Fall2025/blob/gh-pages/files/notebooks/CreditScoreFeatureWeightEstimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install torch
!pip install numpy

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [3]:
# Define a more realistic dataset for credit score prediction
# Features: Payment history (%), Credit utilization (%), Credit history length (years), Types of credit, New credit inquiries
# Labels: Ground truth credit scores (synthetic for demonstration purposes)
data = [
    [95, 30, 10, 3, 2, 750],
    [85, 45, 8, 2, 5, 700],
    [65, 70, 5, 1, 10, 600],
    [90, 40, 15, 4, 3, 720],
    [80, 50, 7, 2, 7, 680],
    [99, 20, 20, 5, 1, 800],
    [70, 60, 4, 1, 9, 640],
    [88, 35, 12, 3, 2, 730],
    [55, 85, 3, 1, 12, 580],
    [92, 25, 18, 4, 4, 760]
]

# Feature names for clarity - the final column is the calculated credit score
feature_names = [
    "Payment History",
    "Credit Utilization",
    "Credit History Length",
    "Types of Credit",
    "New Credit Inquiries"
]

In [4]:
# Split features and labels - the last column is the credit score, the remaining columns are the features that somehow result in that credit score
features = np.array([d[:-1] for d in data], dtype=np.float32)
labels = np.array([d[-1] for d in data], dtype=np.float32).reshape(-1, 1)

# Convert to PyTorch tensors
X = torch.tensor(features)
y = torch.tensor(labels)

In [5]:
# Define a simpler linear regression model without a hidden layer
class SimpleCreditScoreModel(nn.Module):
    def __init__(self):
        super(SimpleCreditScoreModel, self).__init__()
        self.linear = nn.Linear(5, 1)  # Direct mapping from 5 features to 1 output

    def forward(self, x):
        return self.linear(x)


In [6]:
# Initialize the linear model, loss function, and optimizer
model = SimpleCreditScoreModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [7]:
# Training loop for the simpler linear model
epochs = 2000
for epoch in range(epochs):
    # Forward pass
    outputs = model(X)
    loss = criterion(outputs, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss occasionally
    if (epoch + 1) % 200 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [200/2000], Loss: 226.2824
Epoch [400/2000], Loss: 217.0681
Epoch [600/2000], Loss: 210.7635
Epoch [800/2000], Loss: 206.1359
Epoch [1000/2000], Loss: 202.5128
Epoch [1200/2000], Loss: 199.5239
Epoch [1400/2000], Loss: 196.9616
Epoch [1600/2000], Loss: 194.7090
Epoch [1800/2000], Loss: 192.6936
Epoch [2000/2000], Loss: 190.8733


In [8]:
# Get the weights and biases of the linear layer to estimate feature importance
weights = model.linear.weight.data
bias = model.linear.bias.data

In [9]:
# Print the equation for interpretability
print("\nEquation for Feature Influence on Credit Score:")

# Construct the equation string for a linear relationship
equation_terms = []
for i, (weight, feature_name) in enumerate(zip(weights[0], feature_names)):
    equation_terms.append(f"({weight.item():.4f}) * {feature_name}")

equation = " + ".join(equation_terms) + f" + Bias({bias.item():.4f})"
print("Credit Score Estimate = " + equation)

# Compute percentage contribution of each feature
absolute_weights = torch.abs(weights[0])  # Take absolute values to consider magnitude only
total_weight = torch.sum(absolute_weights)
percent_contributions = (absolute_weights / total_weight) * 100  # Calculate percentage contributions

# Print the percentage contribution of each feature
print("\nPercentage Contribution of Each Feature to the Credit Score Estimate:")
for i, (feature_name, contribution) in enumerate(zip(feature_names, percent_contributions)):
    print(f"{feature_name}: {contribution.item():.2f}%")


Equation for Feature Influence on Credit Score:
Credit Score Estimate = (6.9998) * Payment History + (1.7142) * Credit Utilization + (3.0445) * Credit History Length + (-0.1419) * Types of Credit + (2.3980) * New Credit Inquiries + Bias(-0.2168)

Percentage Contribution of Each Feature to the Credit Score Estimate:
Payment History: 48.95%
Credit Utilization: 11.99%
Credit History Length: 21.29%
Types of Credit: 0.99%
New Credit Inquiries: 16.77%


In [11]:
# Prompt user for feature values and calculate credit score

print("\nEnter values for each feature to calculate a credit score estimate:")

# Collect user inputs for each feature
feature_values = []
for feature_name in feature_names:
    value = float(input(f"Enter value for {feature_name}: "))
    feature_values.append(value)

# Convert to a torch tensor for compatibility
x_input = torch.tensor(feature_values, dtype=torch.float32)

# Compute score: linear combination + bias
score = torch.dot(weights[0], x_input) + bias

print("\n--- Credit Score Calculation ---")
equation_terms = [f"({w.item():.4f})*{v:.2f}" for w, v in zip(weights[0], feature_values)]
equation_str = " + ".join(equation_terms) + f" + Bias({bias.item():.4f})"
print("Equation:", equation_str)
print(f"Estimated Credit Score = {score.item():.2f}")



Enter values for each feature to calculate a credit score estimate:
Enter value for Payment History: 65
Enter value for Credit Utilization: 70
Enter value for Credit History Length: 5
Enter value for Types of Credit: 1
Enter value for New Credit Inquiries: 10

--- Credit Score Calculation ---
Equation: (6.9998)*65.00 + (1.7142)*70.00 + (3.0445)*5.00 + (-0.1419)*1.00 + (2.3980)*10.00 + Bias(-0.2168)
Estimated Credit Score = 613.82
